# DPR-To-PIPE Tool

## GUI Usage
- Go to **View**
- Click on **Collapse All Code**
- The notebook will behave like a graphical interface.

---

**Author:** Sarvesh S. Bhogaokar, Master Student (Looking for a PhD)

**Affiliation:** University of Potsdam, Germany  

**Email:**  
- sarvesh1722204@gmail.com  
- bhogaokar@uni-potsdam.de  

**Co-authors:**  
- Dr. Bruno Merin (ESA)

**Acknowledgements:**  
- Dr. Alexis Brandeker (Stockholm University, Sweden)  

**Date:** 02 March 2026  
**Version:** 1.0  

---

### Purpose

This notebook provides a graphical interface (GUI) for:

- Loading CHEOPS L2 SCI_COR lightcurve products  
- Extracting observation metadata  
- Checking existing PIPE database runs  
- Creating structured output directories  
- Running the CHEOPS PIPE backend  
- Managing versioned PIPE processing runs  

The tool is intended for ESA Datalabs users and CHEOPS collaborators.

# What This Notebook Does

This notebook provides a GUI to manage and execute CHEOPS PIPE processing for a single visit.

It allows you to:

- Load a CHEOPS `SCI_COR_Lightcurve` directory
- Automatically extract Visit ID and metadata
- Check for existing PIPE runs
- Select or create an output directory
- Run PIPE with automatic version control

---

# How To Use

1. Select the CHEOPS download folder  
2. Click **Proceed**  
3. (Optional) Check existing PIPE data  
4. Select or create output directory  
5. Click **Run PIPE** and confirm  

---

Output structure created automatically:
---

ℹ️ **Tip:** Collapse all code cells (View → Collapse All Code) to use this notebook as a GUI interface.

In [ ]:
import os
import glob
from astropy.io import fits
import ipywidgets as widgets
from IPython.display import display, clear_output
from ipysplitpanes import SplitPanes
from ipyfilechooser import FileChooser
from ipydatagrid import DataGrid
from pathlib import Path
import pandas as pd
import sys, os
import warnings
import re
from ipywidgets import GridspecLayout
import numpy as np
import matplotlib.pyplot as plt
import time
import subprocess
import threading
if "/media/pipe_script_backend" not in sys.path:
    sys.path.append("/media/pipe_script_backend")


sys.path.insert(0, os.path.abspath("."))


pd.set_option('display.max_colwidth', None)   # show full column content
pd.set_option('display.max_rows', None)       # optional: show all rows
pd.set_option('display.max_columns', None)    # optional: show all columns
pd.set_option('display.width', None)          # auto-detect display width

pd.set_option("display.float_format", "{:.6f}".format)

# warnings.filterwarnings("ignore")


%load_ext autoreload
%autoreload 1

#sys.path.append(os.path.abspath('pipe_script_backend'))

%aimport data_load
%aimport run_pipe_backend




# ---------------------
# STATE
# ---------------------
STATE = {
    "input_dir": None,
    "sci_cor_file": None,
    "metadata": None,
    "visit_id": None,
    "pipe_dir": None
}

# ---------------------
# UI COMPONENTS
# ---------------------

# Phase 1
input_chooser = FileChooser(
    path="/media/home/my_workspace/",
    title='Select CHEOPS Download Directory',
    show_only_dirs=True
)

btn_proceed = widgets.Button(
    description="Proceed",
    button_style="primary"
)

output_area = widgets.Output()







pipe_output_area = widgets.Output()

def build_metadata_section():
    meta_section_title = widgets.HTML("<h3>Observation Metadata</h3>")
    
    # Phase 2 (hidden initially)
    btn_check_pipe = widgets.Button(
        description="Check in PIPE Database",
        button_style="info",
    )

    btn_reprocess = widgets.Button(
        description="Continue / Reprocess",
        button_style="warning",
    )

    
    
    after_cheops_load_hbox = widgets.HBox([btn_check_pipe, btn_reprocess])
    meta_output = widgets.Output()

    with meta_output:
        display(pd.DataFrame([STATE['metadata']]))
    
    container = widgets.VBox([meta_section_title,meta_output,after_cheops_load_hbox],layout=widgets.Layout(
                                border="1px solid #ccc",
                                padding="10px",
                                width="100%"
    
    ))
    
    
    def on_check_pipe(b):
        check_section = build_check_pipe_section()
        pipe_output_area.clear_output()
        with pipe_output_area:
            display(check_section)
            
    def on_reprocess_clicked(b):
        output_section = build_output_directory_section()
        pipe_output_area.clear_output()
        with pipe_output_area:
            display(output_section)

    btn_reprocess.on_click(on_reprocess_clicked)
            
    
    btn_check_pipe.on_click(on_check_pipe)
        
    
    
    return container

    
    
def build_check_pipe_section():

    check_title = widgets.HTML("<h4>Check Existing PIPE Data</h4>")

    pipe_dir_chooser = FileChooser(
        title="Select Mounted PIPE Directory",
        show_only_dirs=True
    )

    btn_default_path = widgets.Button(
        description="Use Default Path",
        button_style="success"
    )

    btn_search_visit = widgets.Button(
        description="Search Visit",
        button_style="primary",
        disabled=True
    )

    btn_still_process = widgets.Button(
        description="Still Process PIPE",
        button_style="warning"
    )
    
    def on_still_process_clicked(b):
        output_section = build_output_directory_section()
        pipe_output_area.clear_output()
        with pipe_output_area:
            display(output_section)

    btn_still_process.on_click(on_still_process_clicked)

    search_output = widgets.Output()

    # -----------------------
    # Default Path Logic
    # -----------------------
    def on_default_clicked(b):
        default_path = "/data/user/che_pipe/Output_lc/visits"
        pipe_dir_chooser.reset(default_path)

        STATE["pipe_dir"] = default_path

        with search_output:
            clear_output()
            print(f"Using default PIPE path:\n{default_path}")

        btn_search_visit.disabled = False

    btn_default_path.on_click(on_default_clicked)

    # -----------------------
    # Manual Selection Logic
    # -----------------------
    def on_path_selected(change):
        if pipe_dir_chooser.selected:
            STATE["pipe_dir"] = pipe_dir_chooser.selected
            btn_search_visit.disabled = False

    pipe_dir_chooser.register_callback(on_path_selected)

    # -----------------------
    # Search Logic
        # -----------------------
    def on_search_clicked(b):

        with search_output:
            clear_output()

            pipe_dir = STATE.get("pipe_dir")
            visit_id = STATE.get("visit_id")

            if not pipe_dir:
                print("Please set PIPE directory first.")
                return

            if not visit_id:
                print("Visit ID not found. Please load observation first.")
                return

            pipe_dir = Path(pipe_dir)

            visit_folder = None

            # ------------------------
            # Locate Visit Folder
            # ------------------------
            for folder in pipe_dir.iterdir():
                if folder.is_dir() and visit_id in folder.name:
                    visit_folder = folder
                    break

            if not visit_folder:
                print("No existing PIPE data found for this visit.")
                display(btn_still_process)
                return

            # ------------------------
            # Inspect Runs
            # ------------------------
            results = []

            for run_folder in visit_folder.iterdir():

                if not run_folder.is_dir():
                    continue

                sa_file = run_folder / f"{visit_id}_L0.5_sa.fits"
                im_file = run_folder / f"{visit_id}_L0.5_im.fits"

                results.append({
                    "Visit ID": visit_id,
                    "Run Folder": run_folder.name,
                    "Subarray (_sa.fits)": sa_file.exists(),
                    "Image (_im.fits)": im_file.exists(),
                    "Run Path": str(run_folder),

                })

            if results:
                df = pd.DataFrame(results)
                display(df)
            else:
                print("Visit folder found, but no run directories detected.")

            display(btn_still_process)
    btn_search_visit.on_click(on_search_clicked)

    # -----------------------
    # Layout
    # -----------------------
    container = widgets.VBox([
        check_title,
        pipe_dir_chooser,
        btn_default_path,
        btn_search_visit,
        search_output
    ], layout=widgets.Layout(
        border="1px solid #ccc",
        padding="10px",
        width="100%"
    ))

    return container


def build_output_directory_section():

    title = widgets.HTML("<h4>Select or Create Output Directory</h4>")

    output_dir_chooser = FileChooser(
        title="Select Parent Directory",
        show_only_dirs=True
    )

    new_folder_text = widgets.Text(
        placeholder="Enter new folder name",
        description="New Folder:"
    )

    btn_create_folder = widgets.Button(
        description="Create Folder",
        button_style="success",
        disabled=True
    )

    btn_confirm_output = widgets.Button(
        description="Confirm Output Directory",
        button_style="primary",
        disabled=True
    )

    btn_run_pipe = widgets.Button(
        description="Run PIPE",
        button_style="danger",
        disabled=True
    )

    confirmation_output = widgets.Output()

    console_output = widgets.Output(
        layout=widgets.Layout(
            height="250px",
            border="1px solid #ccc",
            overflow="auto"
        )
    )

    output_status = widgets.Output()

    # --------------------------------------------------
    # Enable Create Button
    # --------------------------------------------------
    def on_text_change(change):
        btn_create_folder.disabled = not bool(new_folder_text.value.strip())

    new_folder_text.observe(on_text_change, names="value")

    # --------------------------------------------------
    # Create Folder
    # --------------------------------------------------
    def on_create_clicked(b):

        with output_status:
            clear_output()

            parent_dir = output_dir_chooser.selected
            folder_name = new_folder_text.value.strip()

            if not parent_dir:
                print("Please select a parent directory first.")
                return

            new_path = Path(parent_dir) / folder_name

            if new_path.exists():
                print("Folder already exists.")
            else:
                new_path.mkdir(parents=True)
                print(f"Created folder:\n{new_path}")

            output_dir_chooser.reset(str(new_path))
            STATE["output_dir"] = str(new_path)
            btn_confirm_output.disabled = False

    btn_create_folder.on_click(on_create_clicked)

    # --------------------------------------------------
    # Manual Selection
    # --------------------------------------------------
    def on_output_selected(change):
        if output_dir_chooser.selected:
            STATE["output_dir"] = output_dir_chooser.selected
            btn_confirm_output.disabled = False

    output_dir_chooser.register_callback(on_output_selected)

    # --------------------------------------------------
    # Confirm Output Directory
    # --------------------------------------------------
    def on_confirm_clicked(b):

        with output_status:
            clear_output()

            if not STATE.get("output_dir"):
                print("No valid output directory selected.")
                return

            print("Final Output Directory Selected:")
            print(STATE["output_dir"])

            btn_run_pipe.disabled = False

    btn_confirm_output.on_click(on_confirm_clicked)

    # --------------------------------------------------
    # Run PIPE (Synchronous Mode)
    # --------------------------------------------------
    def on_run_clicked(b):

        confirmation_output.clear_output()

        input_dir = STATE.get("input_dir")
        visit_id = STATE.get("visit_id")
        metadata = STATE.get("metadata")
        output_dir = STATE.get("output_dir")

        if not all([input_dir, visit_id, metadata, output_dir]):
            with confirmation_output:
                print("Missing required information before running PIPE.")
            return

        target = metadata.get("Target Name")

        confirm_button = widgets.Button(
            description="Yes, Run",
            button_style="success"
        )

        cancel_button = widgets.Button(
            description="Cancel",
            button_style="warning"
        )

        with confirmation_output:
            display(widgets.HTML(f"""
            <h4>Confirmation Required</h4>
            <b>Input Directory:</b> {input_dir}<br>
            <b>Target:</b> {target}<br>
            <b>Visit ID:</b> {visit_id}<br>
            <b>Output Directory:</b> {output_dir}
            """))
            display(widgets.HBox([confirm_button, cancel_button]))
            


        # --------------------------------------------------
        # Cancel Confirmation
        # --------------------------------------------------
        def on_cancel_confirm(x):
            confirmation_output.clear_output()

        # --------------------------------------------------
        # Confirm and Execute
        # --------------------------------------------------
        def on_confirm_run(x):

            confirmation_output.clear_output()

            input_path = Path(STATE["input_dir"])
            visit = STATE["visit_id"]
            target = STATE["metadata"]["Target Name"]
            parent_output = Path(STATE["output_dir"])

            input_name = input_path.name
            base_output = parent_output / input_name / visit
            base_output.mkdir(parents=True, exist_ok=True)

            # Version finder
            def find_version(outdir_path, limit=10000):
                for version in range(limit):
                    candidate = outdir_path / str(version)
                    if not candidate.exists():
                        return str(version)
                raise Exception("Could not find available output directory")

            version = find_version(base_output)
            run_output_dir = base_output / version
            run_output_dir.mkdir()

            STATE["run_output_dir"] = str(run_output_dir)
            STATE["pipe_version"] = version

            pipe_log_file = parent_output / "pipe_processing_log.csv"
            STATE["pipe_log_file"] = str(pipe_log_file)

            # Lock UI
            input_chooser.disabled = True
            btn_proceed.disabled = True
            btn_run_pipe.disabled = True

            with confirmation_output:
                display(widgets.HTML(f"""
                <h4>PIPE Execution Started</h4>
                <b>Target:</b> {target}<br>
                <b>Visit:</b> {visit}<br>
                <b>Version:</b> {version}<br>
                <b>Run Directory:</b><br>{run_output_dir}
                """))

            # --------------------------------------------------
            # Here we will connect real backend in next step
            # --------------------------------------------------

            with console_output:
                print("PIPE execution would start here...")
                result = run_pipe_backend.run_pipe(
                    input_dir=STATE["input_dir"],
                    run_output_dir=STATE["run_output_dir"],
                    target=STATE["metadata"]["Target Name"],
                    visit=STATE["visit_id"]
                )

        confirm_button.on_click(on_confirm_run)
        cancel_button.on_click(on_cancel_confirm)

    btn_run_pipe.on_click(on_run_clicked)

    # --------------------------------------------------
    # Layout
    # --------------------------------------------------
    container = widgets.VBox([
        title,
        output_dir_chooser,
        widgets.HBox([new_folder_text, btn_create_folder]),
        btn_confirm_output,
        btn_run_pipe,
        confirmation_output,
        console_output,
        output_status
    ])

    return container
    
    

# ---------------------
# INITIAL VISIBILITY CONTROL
# ---------------------

# # Hide PIPE-related widgets initially
# pipe_dir_chooser.layout.display = "none"
# btn_default_pipe.layout.display = "none"
# btn_search_pipe.layout.display = "none"



################ Controller

def on_proceed_clicked(b):

    with output_area:
        clear_output()

        selected_dir = input_chooser.selected

        if not selected_dir:
            print("! Please select a directory.")
            return

        try:
            sci_file = data_load.find_sci_cor_file(selected_dir)
            visit_id = data_load.extract_visit_id_from_filename(sci_file)

            STATE["input_dir"] = selected_dir
            STATE["sci_cor_file"] = sci_file
            STATE["visit_id"] = visit_id
            
            # --- Extract metadata ---
            with fits.open(sci_file) as hdul:
                header = hdul[1].header

                metadata = {
                    "Target Name": header.get("TARGNAME"),
                    "Program ID": header.get("PROG_ID"),
                    "ObsID": header.get("OBSID"),
                    "Visit ID": visit_id,
                    "RA": header.get("RA_TARG"),
                    "DEC": header.get("DEC_TARG"),
                    "CHEOPS Mag": header.get("MAG_CHPS") or header.get("MAG")
                }

            STATE["metadata"] = metadata
            
            metadata_section = build_metadata_section()
            
            display(metadata_section)

            


        except Exception as e:
            print("!!!!!!!!!!!! Error:")
            print(e)


btn_proceed.on_click(on_proceed_clicked)

    
    

# ---------------------
# APP LAYOUT
# ---------------------
# CHEOPS Download section

data_load_heading = widgets.HTML("<h4>CHEOPS Product Directory</h4>")

data_load_vbox = widgets.VBox([input_chooser, btn_proceed], layout=widgets.Layout(
                                border="1px solid #ccc",
                                padding="10px",
                                width="50%"
                                ))




app = widgets.VBox([
    data_load_heading,
    data_load_vbox,
    output_area,
    pipe_output_area
])

display(app)